# Leader Election, Hands-On

Companion notebook for *System Design, Plainly — Season 2, Part 1: Leader Election*.
Three toy models, all runnable: (1) quorum availability arithmetic, (2) the split-vote
simulation that shows why Raft randomizes election timeouts, and (3) fencing tokens —
how a stale leader's writes get rejected after a new leader is elected.

Nothing here is production telemetry. The numbers are this machine's toy-model
measurements, meant to make the intuition checkable.

## 1. Quorum math: how many nodes can you lose and still elect a leader?

A cluster of `n` nodes elects a leader only when a **majority** — `floor(n/2) + 1` —
is up. Two majorities always overlap (pigeonhole principle), so at most one leader
can be elected per term. That overlap is the whole safety argument.

In [ ]:
from math import comb

def cluster_avail(n, p):
    """P(cluster of n nodes, each up with prob p, can still elect a leader)."""
    maj = n // 2 + 1
    return sum(comb(n, k) * p**k * (1 - p)**(n - k) for k in range(maj, n + 1))

for p in (0.99, 0.95):
    print(f"each node {p:.0%} up:")
    for n in (1, 3, 5, 7):
        a = cluster_avail(n, p)
        print(f"  {n}-node cluster: availability {a:.6f} -> "
              f"{(1 - a) * 525600:,.1f} leaderless minutes/year, "
              f"tolerates {(n - 1) // 2} failure(s)")

# sanity: a 1-node cluster is exactly the node's own availability
assert abs(cluster_avail(1, 0.99) - 0.99) < 1e-12
# the striking one: a 3-node cluster of 95%-up nodes beats a single 99%-up node
assert cluster_avail(3, 0.95) > cluster_avail(1, 0.99)
assert cluster_avail(5, 0.99) > cluster_avail(3, 0.99)
print('quorum arithmetic checks out')

## 2. Why the timeout is randomized: the split-vote simulation

If every follower uses the same 200 ms timeout, they all fire at the same instant,
all become candidates, each votes for itself, and nobody reaches a majority.
Randomizing the timeout (Raft's 150–300 ms window) means one node usually fires
first and wins cleanly. Simulation: 20,000 elections on a 5-node cluster; nodes
starting within one 10 ms network round-trip of the earliest count as simultaneous.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

def split_rate(n=5, timeout_range=(150, 300), fixed=None, trials=20000):
    splits = 0
    for _ in range(trials):
        to = np.full(n, float(fixed)) if fixed is not None else rng.uniform(*timeout_range, size=n)
        t0 = to.min()
        cands = np.where(to <= t0 + 10)[0]
        if len(cands) == 1:
            continue
        votes = {c: 1 for c in cands}
        for i in range(n):
            if i not in cands:
                votes[rng.choice(cands)] += 1
        if max(votes.values()) < n // 2 + 1:
            splits += 1
    return splits / trials

fixed = split_rate(fixed=200)
randomized = split_rate()
print(f"fixed 200 ms timeout:    {fixed:.1%} of first attempts split")
print(f"randomized 150-300 ms:   {randomized:.2%} of first attempts split")
print(f"randomization cuts split elections by ~{fixed / max(randomized, 1e-9):.0f}x")

assert fixed > 0.9, 'identical timeouts should almost always split in this toy model'
assert randomized < 0.05, 'jitter should make splits rare'
print('split-vote simulation checks out')

## 3. The leaderless budget: election timeout x how often the leader dies

Detection takes one election timeout; the vote itself is about one network
round-trip. So the leaderless window per failure is roughly the timeout —
and the yearly total is the timeout times the leader-death rate. Compare the
tunings people actually run: Kubernetes Lease (15 s), etcd default (1,000 ms),
the Raft paper's window (150–300 ms).

In [ ]:
scenarios = [
    ('Kubernetes Lease (15 s)', 15.0),
    ('etcd default (1,000 ms)', 1.0),
    ('Raft paper (150-300 ms)', 0.225),
]
for name, timeout in scenarios:
    print(f"{name:28s} 1 death/yr -> {timeout:5.2f} s/yr leaderless | "
          f"1 death/month -> {timeout * 12:6.1f} s/yr leaderless")

# The gap that matters: K8s lease vs etcd default at one leader death a month
assert 15.0 * 12 / (1.0 * 12) == 15.0  # 15x slower failover, same failure rate
print('failover budget checks out: K8s lease costs 15x the leaderless time of etcd default')

## 4. Fencing: the stale leader must not be able to write

Election alone does not prevent split-brain writes. The old leader may be alive
but partitioned — it still thinks it leads. The fix is a **fencing token**: every
leader gets a monotonically increasing term, and the store rejects any write
carrying an older term. Simulate it: leader A (term 6) is partitioned, B wins
term 7, then A's delayed write arrives.

In [ ]:
class Store:
    """A store that only accepts writes from the newest term it has seen."""
    def __init__(self):
        self.term = 0
        self.data = {}

    def write(self, leader, term, key, value):
        if term < self.term:
            return f"REJECTED: {leader} term {term} is stale (store is at term {self.term})"
        self.term = term
        self.data[key] = value
        return f"accepted: {leader} term {term} wrote {key}={value}"

s = Store()
print(s.write('A', 6, 'balance', 100))   # A is leader, term 6
print(s.write('B', 7, 'balance', 90))    # B wins term 7 after A is partitioned
print(s.write('A', 6, 'balance', 0))     # A's delayed write arrives late
print('final balance:', s.data['balance'])

assert s.data['balance'] == 90, 'the stale leader must never overwrite the new leader'
print('fencing checks out: one stale write, zero corruption')